# Train RL-Chess-Engine on Colab (free GPU)

This notebook runs a full self-play training session on Colab's free, dedicated compute — so the engine can train uninterrupted (and on a GPU) instead of competing for memory on a laptop.

**Setup:** `Runtime → Change runtime type → Hardware accelerator → GPU`, then `Runtime → Run all`.

Why this helps: AlphaZero-style training is dominated by *self-play*, and a weak engine only escapes its early "everything is a draw" phase after **many** games. A free laptop CPU can't produce those in reasonable time; a dedicated Colab session can run for hours and use the GPU for network inference.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and dependencies

PyTorch ships with Colab; we only need `python-chess`.

In [ ]:
!git clone https://github.com/Danny-397/RL-Chess-Engine.git

In [ ]:
%cd RL-Chess-Engine
import os, sys
sys.path.insert(0, os.getcwd())
!pip -q install python-chess

## 3. Configure the run

These knobs trade strength for time. The defaults below are a solid "few hours on a free GPU" run that should comfortably beat the random baseline. Scale them up for a stronger engine.

In [ ]:
import torch
from config import Config

cfg = Config()
cfg.training.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- model size: left at the repo default so the checkpoint you download loads
#     locally with NO config changes. Uncomment to train a bigger (stronger,
#     slower) net -- but then set the SAME values locally to load it. ---
# cfg.network.num_channels = 96
# cfg.network.num_residual_blocks = 6

# --- search & self-play: the knobs that matter most for escaping the early
#     "everything draws" phase and actually beating random. ---
cfg.mcts.num_simulations = 120          # deeper search -> better training targets
cfg.training.games_per_iteration = 16   # more (and more decisive) games per round
cfg.training.num_iterations = 40
cfg.training.num_self_play_workers = 1  # self-play inference runs on the GPU
cfg.training.eval_every = 5             # log Elo vs random every 5 iterations
cfg.training.eval_games = 12

print('device :', cfg.resolved_device())
print(f'plan   : {cfg.training.num_iterations} iters x {cfg.training.games_per_iteration} games, '
      f'{cfg.mcts.num_simulations} sims')
print(f'model  : {cfg.network.num_channels} channels x {cfg.network.num_residual_blocks} residual blocks')

## 4. Train

Progress prints per iteration (loss breakdown + periodic Elo vs. random) and is also written to `logs/training.log`. Keep this tab active so Colab doesn't disconnect; for very long runs consider Colab Pro.

In [ ]:
from training import train
model = train(cfg)

## 5. Plot the learning curves

In [ ]:
!python plot_progress.py
from IPython.display import Image
Image('assets/training_progress.png')

## 6. Measure strength against the random baseline

In [ ]:
from evaluation import evaluate_against_random

result = evaluate_against_random(model, cfg, num_games=20, num_simulations=cfg.mcts.num_simulations)
print(result.summary('trained', 'random'))

## 7. See what the engine recommends in the opening

In [ ]:
from chess_game import ChessGame
from analysis import analyze_position

print(analyze_position(model, ChessGame(), cfg, top_n=5).render(max_moves=5))

## 8. Download the trained checkpoint

`train` saves the final model to `checkpoints/best.pt`. Download it, then drop it into your local repo as `checkpoints/example_checkpoint.pt` to play against it:

```bash
python main.py --mode play --checkpoint checkpoints/example_checkpoint.pt
python main.py --mode serve   # or in the browser UI
```

> If you increased the model size above, also set `num_channels` / `num_residual_blocks` to the same values locally (or in the config) so the checkpoint loads into a matching architecture.

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')